# 02 — Light Curve Feature Extraction

This notebook shows how to extract features from supernova light curves
that are useful for classification and follow-up prioritization.

These are the same features that brokers like ALeRCE compute internally,
but understanding them helps you evaluate classifications and build
custom filters for your notification subscriptions.

In [ ]:
from alerce.core import Alerce
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('dark_background')
client = Alerce()

# Pull a bright supernova with many detections
sne = client.query_objects(
    classifier='lc_classifier', class_name='SNIa',
    format='pandas', page_size=20, probability=0.9
)
# Pick the one with the most detections
best = sne.sort_values('ndet', ascending=False).iloc[0]
oid = best['oid']
dets = client.query_detections(oid, format='pandas', sort='mjd')
print(f'Using {oid}: {len(dets)} detections, classified as SNIa with p={best.get("probability", "?")}')

## Feature 1: Rise Time and Decline Rate

Supernovae rise to peak brightness quickly (days to weeks) then decline slowly.
Type Ia SNe have a characteristic rise time of about 17-20 days and decline by
1 magnitude in ~15 days after peak (the "Delta m15" parameter).

These timescales help distinguish SN types from each other and from other transients.

In [ ]:
def compute_light_curve_features(detections_df, band_fid=2):
    """Extract basic light curve features from a single band."""
    band = detections_df[detections_df['fid'] == band_fid].copy()
    if len(band) < 3:
        return {}
    
    band = band.sort_values('mjd')
    
    # Peak brightness (minimum magnitude = brightest)
    peak_idx = band['magpsf'].idxmin()
    peak_mag = band.loc[peak_idx, 'magpsf']
    peak_mjd = band.loc[peak_idx, 'mjd']
    
    # Rise time: first detection to peak
    first_mjd = band['mjd'].min()
    rise_time = peak_mjd - first_mjd
    
    # Decline rate: magnitude change in 15 days after peak
    post_peak = band[band['mjd'] > peak_mjd]
    if len(post_peak) > 0:
        # Find detection closest to peak + 15 days
        target_mjd = peak_mjd + 15
        closest = post_peak.iloc[(post_peak['mjd'] - target_mjd).abs().argsort()[:1]]
        delta_m15 = closest['magpsf'].values[0] - peak_mag
    else:
        delta_m15 = None
    
    # Duration: total time span
    duration = band['mjd'].max() - band['mjd'].min()
    
    # Amplitude: total brightness range
    amplitude = band['magpsf'].max() - band['magpsf'].min()
    
    return {
        'peak_mag': peak_mag,
        'peak_mjd': peak_mjd,
        'rise_time_days': rise_time,
        'delta_m15': delta_m15,
        'duration_days': duration,
        'amplitude_mag': amplitude,
        'n_detections': len(band),
    }

features = compute_light_curve_features(dets, band_fid=2)  # r-band
print('Light curve features (r-band):')
for k, v in features.items():
    if v is not None:
        print(f'  {k:20s} = {v:.2f}' if isinstance(v, float) else f'  {k:20s} = {v}')

## Feature 2: Color Evolution

The g-r color (difference between g-band and r-band magnitudes) tells you
about the temperature of the source. Supernovae start blue (hot) and become
redder as they cool. AGN tend to be bluer and vary erratically.

In [ ]:
# Compute g-r color at matched epochs
g_band = dets[dets['fid'] == 1][['mjd', 'magpsf']].rename(columns={'magpsf': 'g_mag'})
r_band = dets[dets['fid'] == 2][['mjd', 'magpsf']].rename(columns={'magpsf': 'r_mag'})

# Match g and r observations by nearest MJD (within 1 day)
colors = []
for _, g_row in g_band.iterrows():
    time_diffs = (r_band['mjd'] - g_row['mjd']).abs()
    nearest_idx = time_diffs.idxmin()
    if time_diffs[nearest_idx] < 1.0:  # Within 1 day
        colors.append({
            'mjd': g_row['mjd'],
            'g_r': g_row['g_mag'] - r_band.loc[nearest_idx, 'r_mag'],
        })

if colors:
    color_df = pd.DataFrame(colors)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Light curve
    for fid, group in dets.groupby('fid'):
        c = '#4fc3f7' if fid == 1 else '#ef5350'
        ax1.errorbar(group['mjd'], group['magpsf'], yerr=group['sigmapsf'],
                    fmt='o', color=c, label='g' if fid==1 else 'r', ms=4, alpha=0.7, capsize=2)
    ax1.invert_yaxis()
    ax1.set_xlabel('MJD'); ax1.set_ylabel('Magnitude'); ax1.legend()
    ax1.set_title(f'Light Curve: {oid}')
    
    # Color evolution
    ax2.scatter(color_df['mjd'], color_df['g_r'], c='#ffd43b', s=30, alpha=0.8)
    ax2.axhline(0, color='white', alpha=0.2, ls='--')
    ax2.set_xlabel('MJD'); ax2.set_ylabel('g - r color')
    ax2.set_title('Color Evolution (positive = redder)')
    
    plt.tight_layout()
    plt.show()
    
    print(f'Color range: {color_df["g_r"].min():.2f} to {color_df["g_r"].max():.2f}')
    print('Typical SNIa: starts near 0 (blue) and evolves to +0.5 to +1.0 (red)')
else:
    print('Not enough matched g/r observations for color analysis')

## Feature 3: ALeRCE's Own Features

ALeRCE computes dozens of features internally for its classifiers.
You can query them directly and see what the ML model uses.

In [ ]:
# Query ALeRCE's computed features
try:
    features_df = client.query_features(oid, format='pandas')
    if features_df is not None and len(features_df) > 0:
        print(f'ALeRCE computed {len(features_df)} features for {oid}:')
        print(features_df.head(20).to_string())
    else:
        print('No features available from ALeRCE for this object')
except Exception as e:
    print(f'Feature query not available: {e}')

## Next Steps

These features form the basis of alert filtering in Rubin Scout.
When you create a subscription, you're essentially saying:
"Notify me when an object has features X, Y, Z."

For more advanced analysis:
- Fit template light curves (SALT2 for SNIa) to estimate redshift and phase
- Compute Bazin function parameters for parametric light curve modeling
- Use the full ALeRCE probability vector to flag uncertain classifications
  for human review